In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "46d487bb9b699b97c7c5fc5edd866e4d2ad7919b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/46d487bb9b699b97c7c5fc5edd866e4d2ad7919b/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Padding, Stride, and Dilation

An unpadded $k \times k$ convolution shrinks its input by $k - 1$ pixels along
each axis (that section). Across ten $5 \times 5$ layers, a
$240 \times 240$ image shrinks to $200 \times 200$, discarding 30% of its area
along the borders. Yet keeping every layer at full resolution with stride 1 has
the opposite problem: each $3 \times 3$ layer widens a unit's receptive field by
only two pixels, requiring over a hundred layers to span a $224 \times 224$
image. And in dense prediction tasks like semantic segmentation, we want *both*
a wide field of view and full spatial resolution without the quadratic cost of
huge kernels.

Three geometric controls navigate these trade-offs: *padding* surrounds the
input with a border so kernels can center on edge pixels and preserve spatial
dimensions; *stride* steps the window by more than one pixel to downsample
feature maps and accelerate receptive-field growth; and *dilation* spaces kernel
weights apart so a small kernel spans a wider footprint at full resolution.

In [1]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## Padding

### Boundary Pixels

Even before successive layers trim away the perimeter of an image, pixels near
the border are heavily underutilized. the figure counts how many
sliding windows cover each pixel for an unpadded convolution. With a
$k\times k$ kernel, any interior pixel at least $k-1$ steps from every border
participates in $k^2$ windows, whereas an edge pixel appears in at most $k$
windows and a corner pixel in only one. For a $5\times 5$ kernel, an interior
pixel contributes to 25 output locations while a corner pixel contributes to
just one, meaning that an object near the corner barely registers in the next
layer. Without intervention, each layer draws disproportionately from the
center of its input.

![Number of windows that contain each pixel when a $1\times 1$, $2\times 2$, or $3\times 3$ kernel slides over a large input without padding. Interior pixels are covered $k^2$ times, corner pixels once. Each stretched cell stands for all pixels at that distance from the border, however many there are.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-reuse.svg)

### Zero Padding

The simplest remedy is to surround the input with a border of extra filler
pixels (typically zeros) before sliding the kernel across it, a technique known
as *padding*. In the figure, adding one row of zeros to the top
and bottom and one column to the left and right expands a $3\times 3$ input
into a $5\times 5$ grid, allowing a $2\times 2$ kernel to produce a
$4\times 4$ output rather than a $2\times 2$ one. The shaded upper-left output
element combines three padded zeros with the corner of the original input:
$0\times0+0\times1+0\times2+0\times3=0$.

![Two-dimensional cross-correlation with one pixel of zero padding on every side.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-pad.svg)

Following the convention used in deep learning frameworks, we count padding
per side: $p_\textrm{h}$ rows added above and $p_\textrm{h}$ below the input,
and $p_\textrm{w}$ columns added to the left and to the right. The padded
input then has shape
$(n_\textrm{h}+2p_\textrm{h})\times(n_\textrm{w}+2p_\textrm{w})$, yielding
an output of shape

$$(n_\textrm{h}+2p_\textrm{h}-k_\textrm{h}+1)\times(n_\textrm{w}+2p_\textrm{w}-k_\textrm{w}+1).$$

Each row of padding on either side adds one row to the output, and each column
adds one column. Because the kernel removes $k_\textrm{h}-1$ rows and
$k_\textrm{w}-1$ columns, setting $p_\textrm{h}=(k_\textrm{h}-1)/2$ and
$p_\textrm{w}=(k_\textrm{w}-1)/2$ preserves the exact input height and width.
This is by far the most common choice in practice: a stack of shape-preserving
layers maintains the same spatial resolution at every depth, making network
architectures much easier to design and reason about.

Notice that dividing $k-1$ by two explains why CNNs almost always use odd
kernel sizes, such as 1, 3, 5, or 7, paired with paddings of 0, 1, 2, or 3.
An odd kernel has a well-defined center pixel, so symmetric padding places the
window computing output `Y[i, j]` directly centered on input pixel `X[i, j]`,
keeping the input and output grids aligned. For an even kernel, $k_\textrm{h}-1$
is odd, so preserving the spatial dimensions requires asymmetric padding (one
extra row on one side compared to the other) and shifts the output grid by
half a pixel relative to the input.

To verify these output-shape formulas in code, we first define a small helper.
Because built-in convolutional layers expect four-dimensional inputs carrying
batch and channel axes, `comp_conv2d` inserts two singleton dimensions around
a 2D input, applies the layer, and strips those extra dimensions from the
result.

In [2]:
def comp_conv2d(conv2d, X):
    """Apply a convolutional layer to a single-channel 2D input."""
    Y = conv2d(X.reshape((1, 1) + X.shape))  # input (1, 1, h, w)
    return Y.reshape(Y.shape[2:])

For a $3\times 3$ kernel, shape-preserving padding requires $p = (3-1)/2 = 1$
(`padding=1` in code). Applying this layer to an $8\times 8$ input should
leave the spatial shape unchanged at $8\times 8$.

In [3]:
X = torch.rand(size=(8, 8))
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1)
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

### Padding Values

Zero padding implicitly assumes that the image is surrounded by black pixels
outside its frame. More formally, padding specifies a *boundary condition*: a
rule for supplying values beyond the image border where no measurement exists.
To see how zero padding affects the output, let us pad a uniform $5\times 5$
image of all ones with one pixel of zeros on each side and cross-correlate it
with a $3\times 3$ kernel of all ones using `corr2d` from
that section. Each output entry then counts how many true image
pixels fall inside its window.

In [4]:
img = d2l.ones((5, 5))
d2l.corr2d(F.pad(img, (1, 1, 1, 1)), d2l.ones((3, 3)))

tensor([[4., 6., 6., 6., 4.],
        [6., 9., 9., 9., 6.],
        [6., 9., 9., 9., 6.],
        [6., 9., 9., 9., 6.],
        [4., 6., 6., 6., 4.]])

The interior outputs equal 9, edge outputs drop to 6, and corner outputs drop
to 4: even though the original image is completely uniform, the layer sees a
dark frame around it. That artificial frame propagates inward through
subsequent layers, which can learn to use it as an implicit positional cue; at
the same time, boundary artifacts can create blind spots near the edges where
the network responds weakly [@Alsallakh.Kokhlikyan.Miglani.ea.2020]. Two
common alternatives avoid introducing zeros: *reflect padding* mirrors pixels
across the border, while *replicate padding* repeats the border values
outward. Both map a constant image to a constant output, making them popular
in image-generation and restoration networks where border darkening would
appear directly in the generated pixels. In classification networks, however,
zero padding remains the standard choice because it is simple and can be
applied implicitly by fast kernels without allocating a larger padded tensor.

## Stride

### Strided Convolution

Keeping a $224\times 224$ image at full resolution throughout a network is
both expensive and slow to aggregate global context: every channel contains
50,176 spatial positions per layer, and since each $3\times 3$ convolution
expands a unit's view by only two pixels, more than 100 layers would be needed
before a single unit could see the entire image. Stepping the convolution
window across the input in larger increments solves both problems at once,
reducing the spatial resolution and shrinking the depth needed for global
context.

Until now, our sliding window has advanced one pixel at a time across each
row and down each column. We call the step size along the height and width the
*stride*. With vertical stride $s_\textrm{h}$ and horizontal stride
$s_\textrm{w}$, the layer evaluates the kernel only at every $s_\textrm{h}$-th
row and every $s_\textrm{w}$-th column, cutting the number of output elements
by roughly a factor of $s_\textrm{h} s_\textrm{w}$. the figure
illustrates a vertical stride of 3 and a horizontal stride of 2 on the padded
input from the figure (an asymmetric choice that makes the two
axes easy to distinguish). Each shaded output matches the color of the window
that produced it: the blue entry equals
$0\times0+0\times1+1\times2+2\times3=8$ from a window shifted two columns to
the right of the first, and the orange entry equals
$0\times0+6\times1+0\times2+0\times3=6$ from a window shifted three rows down.
Stepping two more columns to the right would push the window past the right
edge of the padded input, leaving two outputs per row.

![Cross-correlation with strides of 3 and 2 for height and width, respectively. Each shaded output has the color of the window that produces it.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-stride.svg)

To determine the output height in general, note that the window starts at row 0
of the padded input and must fit entirely within the
$n_\textrm{h}+2p_\textrm{h}$ padded rows. Its valid starting rows are
$0, s_\textrm{h}, 2s_\textrm{h}, \ldots$ up to
$n_\textrm{h}+2p_\textrm{h}-k_\textrm{h}$. Applying the same reasoning to
the width gives an output shape of

$$\left(\left\lfloor\frac{n_\textrm{h}+2p_\textrm{h}-k_\textrm{h}}{s_\textrm{h}}\right\rfloor+1\right)\times\left(\left\lfloor\frac{n_\textrm{w}+2p_\textrm{w}-k_\textrm{w}}{s_\textrm{w}}\right\rfloor+1\right).$$

Here the floor term counts how many full steps the window can take after its
initial placement, and the $+1$ accounts for that starting position. Checking
the figure yields $\lfloor 3/3\rfloor+1=2$ rows and
$\lfloor 3/2\rfloor+1=2$ columns, and setting $s_\textrm{h}=s_\textrm{w}=1$
recovers our earlier padding formula. When we pair stride with
shape-preserving padding ($2p_\textrm{h}=k_\textrm{h}-1$), the output height
simplifies to
$\lfloor (n_\textrm{h}-1)/s_\textrm{h}\rfloor+1=\lceil n_\textrm{h}/s_\textrm{h}\rceil$:
a stride of $s$ divides the spatial resolution by $s$, rounding up.

A very common downsampling layer, used throughout the ResNet family
(that section), is a $3\times 3$ convolution with padding 1 and
stride 2. Applied to our $8\times 8$ input, it halves both spatial dimensions
to $4\times 4$.

In [5]:
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1, stride=2)
comp_conv2d(conv2d, X).shape

torch.Size([4, 4])

Height and width need not share the same kernel size, padding, or stride; the
output-shape formula applies independently to each axis.

An important special case occurs when the stride equals the kernel size and
padding is zero: the windows then partition the input into non-overlapping
tiles. For instance, a $16\times 16$ convolution with stride 16 splits a
$224\times 224$ image into a $14\times 14$ grid of patches and projects each
patch to a vector of channel outputs. This *patchify* stem is how vision
transformers convert images into sequences of tokens, and several modern CNNs
open with the same operation.

When height and width share the same settings, we drop the subscripts and
simply write $p$ and $s$. Unless specified otherwise, a convolutional layer
defaults to stride 1 and zero padding ($p=0$).


### Stride and the Receptive Field

In that section, we saw that at stride 1 each $k\times k$ layer
expands the receptive field by $k-1$ pixels per axis, producing only linear
growth with depth. Stride accelerates this growth dramatically. After a layer
with stride 2, adjacent entries in its output correspond to locations two
pixels apart in the original image; consequently, a subsequent $3\times 3$
kernel widens the receptive field by $(3-1)\cdot 2=4$ input pixels rather than
2. More generally, a single step in the input to layer $i$ spans
$\prod_{j<i} s_j$ pixels of the original image, so a unit at the top of an
$L$-layer stack with kernel sizes $k_i$ and strides $s_i$ has a receptive
field of width

$$r = 1 + \sum_{i=1}^{L} (k_i - 1) \prod_{j=1}^{i-1} s_j,$$

where the empty product for $i = 1$ is defined as 1. Each layer contributes
the span of its kernel measured in original image pixels; setting all strides
to 1 reduces every term to $k_i-1$, recovering the linear count from
that section. Notice that padding does not enter the formula:
padding determines which output locations exist, not how wide a patch each
unit observes. Comparing two three-layer stacks of $3\times 3$ convolutions,
strides $(1, 1, 1)$ give $r = 1+2+2+2 = 7$, whereas strides $(2, 2, 1)$ more
than double the reach to $r = 1+2+4+8 = 15$.

Because each stride multiplies the effective reach of every layer above it,
the receptive field grows geometrically whenever downsampling continues. After
five stride-2 stages (typical in image classifiers operating on
$224\times 224$ inputs), a single step in the feature map spans 32 input
pixels, enabling top-level units to view the entire image in only a handful of
layers.

## Dilation

### Dilated Convolution

Strided convolutions buy a wide receptive field at the expense of spatial
resolution. For some tasks, however, we want both a wide context and an output
at every input pixel. Simply enlarging the kernel would give a wider view, but
its parameter count and compute cost grow quadratically with kernel size.
Whereas padding and stride control *where* the kernel is placed, dilation
modifies the *spacing of the kernel itself*: a *dilated convolution*
[@yu2016dilated] inserts gaps between adjacent weights so that they are
evaluated $d$ pixels apart, where $d$ is the *dilation* rate. A $k\times k$
kernel with dilation $d$ still contains only $k^2$ learnable weights, yet
spans an *effective kernel size* of

$$k' = k + (k-1)(d-1) = d(k-1)+1$$

pixels along each axis, skipping $d-1$ pixels between consecutive taps. Setting
$d=1$ recovers the ordinary dense convolution. the figure
contrasts a $3\times 3$ kernel at $d=1$ and $d=2$: at dilation 2, the same
nine weights cover a $5\times 5$ footprint by sampling every other pixel,
while still requiring only nine multiplications per output location.

![A $3\times 3$ kernel at dilation 1 and dilation 2. At dilation 2 its nine weights span a $5\times 5$ footprint.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-conv-dilation.svg)

As far as output dimensions are concerned, a dilated kernel behaves identically
to a dense kernel of size $k'$. Replacing $k$ with $k' = d(k-1)+1$ in our
strided formula yields a single unified expression for the output size along
an axis with arbitrary padding, stride, and dilation:

$$\left\lfloor\frac{n+2p-d(k-1)-1}{s}\right\rfloor+1,$$

which applies independently to height and width with their respective
parameters $n$, $k$, $p$, $s$, and $d$. Setting $d=1$ reproduces the strided
formula, and setting $s=1$ and $p=0$ brings us all the way back to $n-k+1$.
On our $8\times 8$ input, a $3\times 3$ kernel with dilation 2 should therefore
produce the same $4\times 4$ output shape as a dense $5\times 5$ kernel.

In [6]:
conv2d = nn.LazyConv2d(1, kernel_size=3, dilation=2)
conv5 = nn.LazyConv2d(1, kernel_size=5)
comp_conv2d(conv2d, X).shape, comp_conv2d(conv5, X).shape

(torch.Size([4, 4]), torch.Size([4, 4]))

### Dilation and the Receptive Field

At stride 1, a dilated layer widens the receptive field by $k'-1 = d(k-1)$
pixels, so the equation still holds once each $k_i$ is replaced
by its effective size $k'_i$. If we double the dilation rate at each layer,
$d_i = 1, 2, 4, \ldots, 2^{L-1}$, a stack of $L$ layers with $3\times 3$
kernels attains a receptive field of width
$1 + 2(1 + 2 + \cdots + 2^{L-1}) = 2^{L+1}-1$. Just like a strided stack, its
receptive field grows exponentially with depth, yet every layer retains the
full spatial resolution of the input.

the figure illustrates these two strategies side by side.
Panel (a) shows our strided stack with strides $(2, 2, 1)$: the three layers
add $2$, $4$, and $8$ pixels to the top unit's view, reaching 15 input pixels
in total. Panel (b) achieves the exact same 15-pixel receptive field using
stride 1 and dilations $(1, 2, 4)$, wiring the top unit to the input through
the identical pattern of connections. The sole difference is spatial density:
the strided stack retains only every fourth spatial position, whereas the
dilated stack evaluates all of them. Indeed, without padding and with shared
weights, the strided stack's output is simply every fourth entry of the
dilated stack's output, even when elementwise nonlinearities are applied after
every layer. That makes dilated convolutions especially valuable for *dense
prediction* tasks such as semantic segmentation, where we must assign a class
label to every pixel while still drawing on broad surrounding context
[@yu2016dilated]. The tradeoff is computation and memory, because every
layer must process full-resolution feature maps; an alternative is to
downsample with strides and later recover spatial resolution using learned
upsampling layers such as the transposed convolution of
that section.

![Receptive field of the top unit of three stacked layers with kernel size 3, in one dimension. Cells are the positions each layer computes, shaded cells those the top unit depends on, and lines the kernel's taps. (a) Strides 2, 2, 1: layer 1 keeps every second position, layers 2 and 3 every fourth. (b) Stride 1 with dilations 1, 2, 4: every layer keeps all positions. Both top units see the same 15 input pixels.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-rf-towers.svg)

So far, every example in this section has operated on a single-channel input
with a single kernel. Real-world images contain multiple color channels, and
useful hidden layers detect dozens or hundreds of distinct patterns at once.
We lift both restrictions in that section.

## Summary

Without padding, a convolution shrinks its input by $k-1$ pixels along each
axis and samples border pixels in far fewer windows than interior ones. Adding
$(k-1)/2$ pixels of padding to each side of an odd kernel preserves the input
shape and keeps each output centered on its corresponding input pixel, though
zero padding introduces a boundary condition that deep layers can detect.
Using a stride $s > 1$ evaluates only every $s$-th window position, reducing
spatial resolution and computation by roughly a factor of $s$ per axis while
causing the receptive field of higher layers to grow geometrically. Finally,
dilation $d$ spaces a kernel's weights across an effective footprint of
$d(k-1)+1$, expanding the receptive field exponentially at full resolution
without adding parameters. All three mechanisms are captured by a single
output-shape formula, the equation.

## Exercises

1. [code] **Mirror padding.** Mirror padding, also called reflect padding,
   extends a tensor by reflecting its entries about the border without
   repeating the border entry: padding the vector $(1, 2, 3)$ by one entry
   on each side gives $(2, 1, 2, 3, 2)$.
    1. Implement mirror padding by one entry on every side of a
       two-dimensional tensor, and check the result by hand on a
       $3 \times 3$ tensor with distinct entries.
    1. Repeat the computation of that section with
       mirror padding instead of zero padding, and explain the result.
    1. Take the $5\times 5$ ramp $x_{ij} = j$ and the $3\times 3$ kernel
       whose entries all equal $\frac{1}{9}$. Show that in the interior the
       output equals the input. Compute the outputs in the first and last
       columns under zero padding and under mirror padding, and find a way
       to pad that reproduces the ramp exactly.

2. **Dilation and gridding.** A network stacks four $3 \times 3$
   convolutions with stride 1. The doubling dilation schedule
   $(1, 2, 4, 8)$ gives one output element a receptive field of side $31$.
   Compare it with the constant schedule $(4, 4, 4, 4)$.
    1. Use the equation, with each kernel replaced by its
       effective size, to compute the receptive field of the constant
       schedule.
    1. For each schedule, determine which pixels inside the receptive
       field the output actually depends on, and count them. A schedule
       whose output depends only on a sparse lattice within its receptive
       field is said to exhibit *gridding*.
    1. Give one reason gridding is harmful for dense prediction tasks such
       as segmentation.
    1. ● State a condition on the dilations $d_1, \ldots, d_L$ of a stack
       of $3 \times 3$ convolutions with stride 1 that guarantees that the
       output depends on every pixel of its receptive field, and show that
       the condition suffices.

3. **Strides versus dilations.** Consider the two one-dimensional stacks of
   the figure, without padding, with kernels
   $\mathbf{w}_1, \mathbf{w}_2, \mathbf{w}_3$ of size 3 shared by both
   stacks and an elementwise nonlinearity after every layer.
    1. Write the output of each stack at position $i$ as a function of the
       input, and show that the output of the strided stack at position $i$
       equals the output of the dilated stack at position $4i$.
    1. For an input of length $n$, count the multiplications each stack
       performs, ignoring the border.
    1. Generalize the first part: which dilations reproduce a stack with
       strides $s_1, \ldots, s_L$?

4. **Shapes through a stem.** A $224\times 224$ image passes through four
   layers: a $7\times 7$ convolution with padding 3 and stride 2, a
   $3\times 3$ convolution with padding 1 and stride 2, and two
   $3\times 3$ convolutions with padding 1 and stride 1.
    1. Give the height of the output of every layer.
    1. Use the equation to compute the receptive field of a
       unit of the last layer. What fraction of the image's width does it
       see?
    1. Replace the first two layers by a single $4\times 4$ convolution
       with stride 4 and no padding, the patchify layer of this section.
       Give the output height and the receptive field again. Do neighboring
       windows of the first layer still overlap?

[Discussions](https://d2l.discourse.group/t/68)